# Проверка decoder-only претрейна

Notebook запускается из корня репозитория, где находится `model/model.py`. Декодер вызывается с `encoder_hidden_states=None`, поэтому cross-attention должен быть полностью пропущен.

In [6]:
%pip install -q "flash-linear-attention[cuda]==0.5.2" "transformers==5.13.1"

In [7]:
%cd /content

!git clone https://github.com/SummerSchoolSPBU2026/whisper-kda-asr.git
%cd /content/whisper-kda-asr

/content
fatal: destination path 'whisper-kda-asr' already exists and is not an empty directory.
/content/whisper-kda-asr


In [8]:
import copy
import torch
from transformers import AutoProcessor

from model import (
    KDACrossAttentionDecoder,
    create_kda_config,
)

if not torch.cuda.is_available():
    raise RuntimeError("Для проверки требуется CUDA GPU")

device = torch.device("cuda")

processor = AutoProcessor.from_pretrained(
    "openai/whisper-tiny"
)
processor.tokenizer.set_prefix_tokens(
    language="russian",
    task="transcribe",
    predict_timestamps=False,
)

config = copy.deepcopy(
    create_kda_config(processor.tokenizer)
)
config.is_encoder_decoder = False
config.use_cache = False

decoder = KDACrossAttentionDecoder(config).to(device)
decoder.train()

print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


## Заморозка и контроль cross-attention

Hooks немедленно остановят тест, если cross-attention или его norm будут вызваны.

In [9]:
for layer in decoder.model.layers:
    layer.cross_attn.requires_grad_(False)
    layer.cross_attn_norm.requires_grad_(False)

trainable_parameters = [
    parameter
    for parameter in decoder.parameters()
    if parameter.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=1e-3,
    weight_decay=0.01,
)

cross_before = {
    name: parameter.detach().cpu().clone()
    for name, parameter in decoder.named_parameters()
    if "cross_attn" in name
}
kda_before = {
    name: parameter.detach().cpu().clone()
    for name, parameter in decoder.named_parameters()
    if ".attn." in name
}

def fail_if_called(module, args):
    raise AssertionError(
        f"{module.__class__.__name__} вызван в decoder-only режиме"
    )

hooks = []
for layer in decoder.model.layers:
    hooks.append(
        layer.cross_attn.register_forward_pre_hook(
            fail_if_called
        )
    )
    hooks.append(
        layer.cross_attn_norm.register_forward_pre_hook(
            fail_if_called
        )
    )

## Один LM training step

In [11]:
batch_size = 2
sequence_length = 64

input_ids = torch.randint(
    low=0,
    high=config.vocab_size,
    size=(batch_size, sequence_length),
    dtype=torch.long,
    device=device,
)
attention_mask = torch.ones_like(input_ids)
labels = input_ids.clone()

optimizer.zero_grad(set_to_none=True)

try:
    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
    ):
        outputs = decoder(
            input_ids=input_ids,
            encoder_hidden_states=None,
            decoder_attention_mask=attention_mask,
            labels=labels,
            use_cache=False,
        )

    loss = outputs.loss
    assert loss is not None
    assert torch.isfinite(loss)

    loss.backward()

    nonfinite_gradients = [
        name
        for name, parameter in decoder.named_parameters()
        if parameter.grad is not None
        and not torch.isfinite(parameter.grad).all()
    ]
    assert not nonfinite_gradients, (
        "Обнаружены NaN/inf в градиентах: "
        f"{nonfinite_gradients[:10]}"
    )

    assert all(
        parameter.grad is None
        for layer in decoder.model.layers
        for module in (layer.cross_attn, layer.cross_attn_norm)
        for parameter in module.parameters()
    )

    assert any(
        parameter.grad is not None
        and torch.isfinite(parameter.grad).all()
        for layer in decoder.model.layers
        for parameter in layer.attn.parameters()
    )

    assert any(
        parameter.grad is not None
        for layer in decoder.model.layers
        for parameter in layer.mlp.parameters()
    )

    assert decoder.model.embeddings.weight.grad is not None

    grad_norm = torch.nn.utils.clip_grad_norm_(
        trainable_parameters,
        max_norm=1.0,
        error_if_nonfinite=True,
    )
    optimizer.step()
finally:
    for hook in hooks:
        hook.remove()

cross_unchanged = all(
    torch.equal(
        cross_before[name],
        parameter.detach().cpu(),
    )
    for name, parameter in decoder.named_parameters()
    if name in cross_before
)

kda_changed = any(
    not torch.equal(
        kda_before[name],
        parameter.detach().cpu(),
    )
    for name, parameter in decoder.named_parameters()
    if name in kda_before
)

assert cross_unchanged
assert kda_changed

print("Logits shape:", outputs.logits.shape)
print("Loss:", loss.item())
print("Gradient norm:", grad_norm.item())
print("Cross-attention не вызывался и не изменился: OK")
print("KDA, MLP и embeddings обучаются: OK")
print("Decoder-only pretrain check: OK")

Logits shape: torch.Size([2, 64, 51865])
Loss: 10.988472938537598
Gradient norm: 14.36296558380127
Cross-attention не вызывался и не изменился: OK
KDA, MLP и embeddings обучаются: OK
Decoder-only pretrain check: OK
